# 🧠 Fine-tuning Qwen2.5-3B-Instruct con Prompt OSTH
## Clasificación de Humor en Español — Experimento 2 (Clasificador)

### Motivación

En el Experimento 1 comprobamos que **Qwen2.5-3B base** aporta muy poco al ensamble (`w_osth=0.10`) porque:
- Sus `humor_score` tienen baja varianza (concentrados en ~0.5)
- No conoce el humor latinoamericano/español de Twitter
- Aplica OSTH de forma genérica sin calibrar para el dataset

### Estrategia de este notebook

Fine-tuneamos Qwen2.5-3B con **QLoRA** usando el prompt OSTH simplificado como system prompt.
Esto hace que Qwen aprenda a **razonar con marcos OSTH** antes de clasificar, en lugar de solo etiquetar.

### Diferencias clave vs el fine-tuning de Gemma

| Aspecto | Gemma-3-4B | Qwen2.5-3B (este notebook) |
|---|---|---|
| Chat template | `<start_of_turn>` | `<\|im_start\|>` |
| Response template | `<start_of_turn>model\n` | `<\|im_start\|>assistant\n` |
| System prompt | Lingüística general | Marco OSTH explícito |
| Arquitectura | Gemma-3 | Qwen2.5 |
| Objetivo en ensamble | Clasificador principal | Clasificador con sesgo OSTH |

## ── Sección 0: Instalación ───────────────────────────────────────────────────

In [ ]:
!pip install -q \
    transformers>=4.47.0 \
    datasets==3.2.0 \
    peft==0.14.0 \
    trl==0.13.0 \
    bitsandbytes>=0.46.1 \
    accelerate==1.2.1 \
    scikit-learn \
    evaluate

!pip install -q --upgrade transformers accelerate safetensors

print('✅ Dependencias instaladas')

## ── Sección 1: Autenticación y verificación de GPU ──────────────────────────

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('✅ Autenticado con Colab Secrets')
except Exception:
    login()

In [ ]:
import torch

print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('❌ Se requiere GPU. Cambia el runtime a GPU en Colab.')

## ── Sección 2: Configuración central ───────────────────────────────────────

In [ ]:
import os

CFG = dict(
    # ── Modelo ───────────────────────────────────────────────────────────────
    model_id        = 'Qwen/Qwen2.5-3B-Instruct',

    # ── Datos (mismos archivos que el pipeline Gemma) ────────────────────────
    train_file      = 'dataset_humor_train.json',
    val_file        = None,
    test_file       = 'dataset_humor_test.json',
    text_col        = 'text',
    label_col       = 'klass',
    label_map       = {1: 'Sí', 0: 'No'},

    # ── QLoRA ────────────────────────────────────────────────────────────────
    # Qwen2.5-3B tiene menos capas que Gemma-3-4B → lora_r=16 es suficiente
    # y ahorra VRAM para un batch_size mayor
    lora_r          = 16,
    lora_alpha      = 32,
    lora_dropout    = 0.05,
    target_modules  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],

    # ── Entrenamiento ────────────────────────────────────────────────────────
    # Qwen usa 3B parámetros vs 4B de Gemma → podemos subir batch_size a 16
    max_seq_len     = 256,   # el prompt OSTH es más largo que el de Gemma
    batch_size      = 8,
    grad_accum      = 4,     # effective batch = 32
    epochs          = 5,
    lr              = 2e-4,  # Qwen converge mejor con lr ligeramente mayor
    warmup_ratio    = 0.08,
    weight_decay    = 0.01,
    seed            = 42,

    # ── Salida ───────────────────────────────────────────────────────────────
    output_dir      = './qwen3b-osth-humor-qlora',
)

os.makedirs(CFG['output_dir'], exist_ok=True)
print('✅ Configuración cargada:')
for k, v in CFG.items():
    print(f'  {k:20s} = {v}')

## ── Sección 3: Carga y split del dataset ───────────────────────────────────

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

df = pd.read_json(CFG['train_file'], lines=True)
X  = df[CFG['text_col']].values
y  = df[CFG['label_col']].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size    = 0.10,
    random_state = CFG['seed'],
    stratify     = y,
)

train_df = pd.DataFrame({CFG['text_col']: X_train, CFG['label_col']: y_train})
val_df   = pd.DataFrame({CFG['text_col']: X_val,   CFG['label_col']: y_val})

raw = DatasetDict({
    'train':      Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
})

if CFG['test_file']:
    test_df   = pd.read_json(CFG['test_file'], lines=True)
    raw['test'] = Dataset.from_pandas(test_df)

print(f'Train:      {len(raw["train"]):,} ejemplos')
print(f'Validation: {len(raw["validation"]):,} ejemplos')
if 'test' in raw:
    print(f'Test:       {len(raw["test"]):,} ejemplos')

In [ ]:
# ── Distribución de clases y class weights ────────────────────────────────────
labels_train = raw['train'][CFG['label_col']]
unique, counts = np.unique(labels_train, return_counts=True)

print(f'Train: {len(raw["train"]):,} | Val: {len(raw["validation"]):,}')
print('Distribución train:')
for u, c in zip(unique, counts):
    print(f'   Clase {u} ({CFG["label_map"].get(u, u)}): {c:,} ({100*c/len(labels_train):.1f}%)')

class_counts  = dict(zip(unique, counts))
total         = sum(class_counts.values())
CLASS_WEIGHTS = {k: total / (len(class_counts) * v) for k, v in class_counts.items()}
print(f'\nClass weights: {CLASS_WEIGHTS}')

## ── Sección 4: Tokenizador y template de prompt OSTH ───────────────────────

### Diferencia crítica: Qwen usa `<|im_start|>` en lugar de `<start_of_turn>`

El `RESPONSE_TEMPLATE` debe coincidir **exactamente** con lo que produce
`apply_chat_template` para el turno del assistant en Qwen2.5.
Si no coincide, el `DataCollatorForCompletionOnlyLM` no encuentra dónde
empiezan las respuestas y entrena sobre todo el prompt (incluyendo el input),
lo que produce un modelo que memoriza prompts en lugar de aprender a clasificar.

In [ ]:
from transformers import AutoTokenizer

print('Cargando tokenizador Qwen2.5-3B...')
tokenizer = AutoTokenizer.from_pretrained(CFG['model_id'])
tokenizer.padding_side = 'left'   # crítico para generación correcta
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Detectar automáticamente el response template de Qwen ────────────────────
# Generamos un ejemplo dummy para ver el formato exacto del chat template
dummy_messages = [
    {'role': 'system',    'content': 'eres un asistente'},
    {'role': 'user',      'content': 'hola'},
    {'role': 'assistant', 'content': 'hola'},
]
dummy_prompt = tokenizer.apply_chat_template(
    dummy_messages, tokenize=False, add_generation_prompt=False
)
print('\nFormato del chat template de Qwen2.5:')
print(repr(dummy_prompt))
print()

# El response template es el prefijo que abre el turno del assistant
# En Qwen2.5 es: '<|im_start|>assistant\n'
RESPONSE_TEMPLATE = '<|im_start|>assistant\n'
assert RESPONSE_TEMPLATE in dummy_prompt, (
    f'ERROR: RESPONSE_TEMPLATE no encontrado en el chat template.\n'
    f'Prompt dummy: {repr(dummy_prompt)}\n'
    f'Ajusta RESPONSE_TEMPLATE manualmente.'
)
print(f'✅ RESPONSE_TEMPLATE verificado: {repr(RESPONSE_TEMPLATE)}')

In [ ]:
# ── System prompt OSTH ───────────────────────────────────────────────────────
# Versión simplificada para fine-tuning: orienta el razonamiento sin
# pedir JSON (eso se reserva para el Experimento 2 de extracción de features).
# El modelo aprende a clasificar CON marco conceptual OSTH.

OSTH_SYSTEM_PROMPT = """Eres un experto en lingüística computacional del humor \
especializado en la Teoría Semántica Ontológica del Humor (OSTH).
Para clasificar un tweet como humorístico, busca:
- Un cambio de guion semántico (script switch): el texto empieza en un contexto \
y termina en uno opuesto o inesperado.
- Un punto de disparo (punch line): palabra o frase que activa el cambio.
- Una oposición de tipo normal/anormal, real/irreal o posible/imposible.
Si detectas estos mecanismos, el tweet contiene humor.
Responde ÚNICAMENTE con la palabra 'Sí' (contiene humor) o 'No' (no contiene humor)."""


def make_prompt(tweet: str, include_answer: bool = False, label: int = None) -> str:
    """
    Genera el prompt usando el chat template nativo de Qwen2.5.
    - include_answer=True  → para entrenamiento (incluye la respuesta)
    - include_answer=False → para inferencia (genera a partir del prompt)
    """
    messages = [
        {'role': 'system', 'content': OSTH_SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Tweet a analizar: "{tweet}"'},
    ]
    if include_answer and label is not None:
        answer = CFG['label_map'].get(int(label), 'No')
        messages.append({'role': 'assistant', 'content': answer})
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    else:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


def make_training_example(sample: dict) -> dict:
    label_value = sample.get(CFG['label_col'], None)
    return {'text': make_prompt(
        sample[CFG['text_col']],
        include_answer = True,
        label          = label_value,
    )}


# ── Aplicar template a todos los splits ──────────────────────────────────────
print('Mapeando splits...')
dataset = raw.map(make_training_example, remove_columns=raw['train'].column_names)

# ── Verificación del prompt ───────────────────────────────────────────────────
print('\nEjemplo de prompt de entrenamiento:')
print('─' * 60)
print(dataset['train'][0]['text'])
print('─' * 60)

# Verificar que el response template está presente
example = dataset['train'][0]['text']
assert RESPONSE_TEMPLATE in example, (
    '❌ RESPONSE_TEMPLATE no aparece en el ejemplo de entrenamiento.'
)
print(f'✅ RESPONSE_TEMPLATE presente en el ejemplo de entrenamiento')

# Verificar longitud de tokens
tokens = tokenizer(example, return_tensors='pt')
print(f'Tokens en ejemplo: {tokens["input_ids"].shape[1]} (max_seq_len={CFG["max_seq_len"]})')
if tokens['input_ids'].shape[1] > CFG['max_seq_len']:
    print('⚠️  El ejemplo supera max_seq_len — considera aumentar a 320')

In [ ]:
# ── Análisis de longitud de tokens en el dataset completo ────────────────────
print('Analizando longitud de tokens en el dataset de entrenamiento...')

lengths = [
    tokenizer(ex, return_tensors='pt')['input_ids'].shape[1]
    for ex in dataset['train']['text'][:200]   # muestra de 200
]
lengths = np.array(lengths)

print(f'  Media:    {lengths.mean():.0f} tokens')
print(f'  Mediana:  {np.median(lengths):.0f} tokens')
print(f'  P95:      {np.percentile(lengths, 95):.0f} tokens')
print(f'  Máximo:   {lengths.max()} tokens')
print(f'  > max_seq_len ({CFG["max_seq_len"]}): {(lengths > CFG["max_seq_len"]).mean()*100:.1f}%')

# Si más del 5% de ejemplos supera max_seq_len, ajustamos automáticamente
p99 = int(np.percentile(lengths, 99))
if p99 > CFG['max_seq_len']:
    print(f'\n⚠️  Ajustando max_seq_len de {CFG["max_seq_len"]} → {p99} (P99)')
    CFG['max_seq_len'] = p99
else:
    print(f'\n✅ max_seq_len={CFG["max_seq_len"]} cubre el P99 ({p99} tokens)')

## ── Sección 5: Carga del modelo base con QLoRA ──────────────────────────────

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print('Cargando Qwen2.5-3B-Instruct con cuantización NF4...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'],
    quantization_config  = bnb_config,
    device_map           = 'auto',
    torch_dtype          = torch.bfloat16,
    attn_implementation  = 'eager',   # flash_attention_2 si la GPU lo soporta
)
model.config.use_cache = False   # necesario para gradient checkpointing

mem = torch.cuda.memory_allocated() / 1e9
print(f'\n✅ Modelo cargado — VRAM usada: {mem:.2f} GB')
print(f'   Parámetros totales: {sum(p.numel() for p in model.parameters()):,}')

## ── Sección 6: Configuración de LoRA ───────────────────────────────────────

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r              = CFG['lora_r'],
    lora_alpha     = CFG['lora_alpha'],
    target_modules = CFG['target_modules'],
    lora_dropout   = CFG['lora_dropout'],
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros totales:      {total_params:,}')
print(f'Parámetros entrenables:  {trainable_params:,} ({100*trainable_params/total_params:.2f}%)')
print(f'lora_r={CFG["lora_r"]} | lora_alpha={CFG["lora_alpha"]} | dropout={CFG["lora_dropout"]}')

## ── Sección 7: Entrenamiento con SFTTrainer ─────────────────────────────────

In [ ]:
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from torch.utils.data import WeightedRandomSampler

# ── Collator: solo penaliza los tokens de respuesta (Sí / No) ────────────────
collator = DataCollatorForCompletionOnlyLM(
    response_template = RESPONSE_TEMPLATE,
    tokenizer         = tokenizer,
    mlm               = False,
)

# ── WeightedRandomSampler para balancear clases ───────────────────────────────
sample_weights = [
    CLASS_WEIGHTS[int(lbl)]
    for lbl in raw['train'][CFG['label_col']]
]
sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True,
)

# ── Argumentos de entrenamiento ───────────────────────────────────────────────
training_args = SFTConfig(
    output_dir                    = CFG['output_dir'],
    num_train_epochs              = CFG['epochs'],
    per_device_train_batch_size   = CFG['batch_size'],
    per_device_eval_batch_size    = CFG['batch_size'],
    gradient_accumulation_steps   = CFG['grad_accum'],
    optim                         = 'paged_adamw_8bit',
    learning_rate                 = CFG['lr'],
    lr_scheduler_type             = 'cosine',
    warmup_ratio                  = CFG['warmup_ratio'],
    weight_decay                  = CFG['weight_decay'],
    bf16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    max_seq_length                = CFG['max_seq_len'],
    packing                       = False,
    dataset_text_field            = 'text',
    eval_strategy                 = 'epoch',
    save_strategy                 = 'epoch',
    load_best_model_at_end        = True,
    metric_for_best_model         = 'eval_loss',
    greater_is_better             = False,
    save_total_limit              = 2,
    logging_steps                 = 50,
    report_to                     = 'none',
    seed                          = CFG['seed'],
)

trainer = SFTTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = dataset['train'],
    eval_dataset     = dataset['validation'],
    processing_class = tokenizer,
    data_collator    = collator,
)

print('✅ Trainer configurado')
print(f'   RESPONSE_TEMPLATE   = {repr(RESPONSE_TEMPLATE)}')
print(f'   padding_side        = {tokenizer.padding_side}')
print(f'   packing             = False')
print(f'   lora_r              = {CFG["lora_r"]}')
print(f'   max_seq_len         = {CFG["max_seq_len"]}')
print(f'   class_weights       = {CLASS_WEIGHTS}')

In [ ]:
# ── Entrenamiento ─────────────────────────────────────────────────────────────
print('Iniciando entrenamiento...')
train_result = trainer.train()

print('\n── Métricas de entrenamiento ──────────────────────────────')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v:.4f}')

## ── Sección 8: Inferencia y evaluación ─────────────────────────────────────

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

# ── Encontrar IDs de token para 'Sí' y 'No' en Qwen ─────────────────────────
SI_IDS = tokenizer.encode('Sí', add_special_tokens=False)
NO_IDS = tokenizer.encode('No', add_special_tokens=False)

print(f'Tokens para "Sí": {SI_IDS} → {[tokenizer.decode([t]) for t in SI_IDS]}')
print(f'Tokens para "No": {NO_IDS} → {[tokenizer.decode([t]) for t in NO_IDS]}')

# Si 'Sí' se tokeniza en múltiples tokens, usamos el primero como señal
SI_TOKEN_ID = SI_IDS[0]
NO_TOKEN_ID = NO_IDS[0]

# Verificar que son diferentes
assert SI_TOKEN_ID != NO_TOKEN_ID, '❌ Sí y No comparten el mismo token ID'
print(f'\n✅ SI_TOKEN_ID={SI_TOKEN_ID} | NO_TOKEN_ID={NO_TOKEN_ID}')

In [ ]:
def predict_batch(tweets: list, batch_size: int = 16) -> tuple:
    """
    Inferencia por comparación de logits (Sí vs No) en la posición [-1].
    Más rápido que generación completa y suficiente para clasificación binaria.
    Devuelve (predicciones_binarias, scores_prob_Sí).
    """
    model.eval()
    predictions, si_scores = [], []

    for i in range(0, len(tweets), batch_size):
        batch   = tweets[i: i + batch_size]
        prompts = [make_prompt(t, include_answer=False) for t in batch]

        inputs = tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = CFG['max_seq_len'],
        ).to(model.device)

        with torch.no_grad():
            outputs     = model(**inputs)
            last_logits = outputs.logits[:, -1, :]        # (B, vocab)
            si_logit    = last_logits[:, SI_TOKEN_ID]      # (B,)
            no_logit    = last_logits[:, NO_TOKEN_ID]      # (B,)

            # Probabilidad suavizada de 'Sí'
            p_si = torch.softmax(
                torch.stack([no_logit, si_logit], dim=1), dim=1
            )[:, 1]

            predictions.extend((si_logit > no_logit).int().cpu().tolist())
            si_scores.extend(p_si.cpu().tolist())

    return predictions, si_scores


print('✅ Función predict_batch definida')

In [ ]:
# ── Evaluación en validación ──────────────────────────────────────────────────
print('Evaluando en validación...')
val_tweets = list(raw['validation'][CFG['text_col']])
val_labels = np.array(raw['validation'][CFG['label_col']])

val_preds, val_scores = predict_batch(val_tweets)
val_scores_np = np.array(val_scores)

print('\n── Resultados con umbral 0.5 ──────────────────────────────')
print(classification_report(val_labels, val_preds,
                             target_names=['No humor', 'Humor']))
print('Matriz de Confusión:')
print(confusion_matrix(val_labels, val_preds))

In [ ]:
# ── Búsqueda del umbral óptimo en validación ──────────────────────────────────
best_thresh, best_f1 = 0.5, 0.0

for thresh in np.linspace(0.1, 0.9, 81):
    preds_t = (val_scores_np >= thresh).astype(int)
    f1_t    = f1_score(val_labels, preds_t, average='macro', zero_division=0)
    if f1_t > best_f1:
        best_f1, best_thresh = f1_t, thresh

val_preds_calibrated = (val_scores_np >= best_thresh).astype(int)

print(f'Umbral default  (0.50): F1 Macro = {f1_score(val_labels, val_preds, average="macro"):.4f}')
print(f'Umbral calibrado({best_thresh:.2f}): F1 Macro = {best_f1:.4f}  ← usar este en test')
print('\n── Con umbral calibrado ──────────────────────────────────────')
print(classification_report(val_labels, val_preds_calibrated,
                             target_names=['No humor', 'Humor']))

In [ ]:
# ── Distribución de scores — diagnóstico de calibración ──────────────────────
# Comparar con los scores del Experimento 1 (Qwen base)
# Si std > 0.20, el modelo está bien calibrado
print('── Distribución de scores Qwen fine-tuned ───────────────────')
print(f'  Media:    {val_scores_np.mean():.3f}')
print(f'  Std:      {val_scores_np.std():.3f}   ← objetivo > 0.20 (vs base ~0.10)')
print(f'  P10:      {np.percentile(val_scores_np, 10):.3f}')
print(f'  P50:      {np.percentile(val_scores_np, 50):.3f}')
print(f'  P90:      {np.percentile(val_scores_np, 90):.3f}')

# Separar distribuciones por clase real
humor_scores    = val_scores_np[val_labels == 1]
no_humor_scores = val_scores_np[val_labels == 0]
print(f'\n  Clase Humor    → media score: {humor_scores.mean():.3f}')
print(f'  Clase No humor → media score: {no_humor_scores.mean():.3f}')
print(f'  Separación:                   {humor_scores.mean() - no_humor_scores.mean():.3f}')
print(f'  (cuanto mayor, mejor discrimina el modelo)')

## ── Sección 9: Análisis de errores ─────────────────────────────────────────

In [ ]:
val_df_err = pd.DataFrame({
    'tweet'    : val_tweets,
    'true'     : val_labels,
    'pred'     : val_preds_calibrated,
    'score_si' : val_scores,
})
val_df_err['error']      = val_df_err['true'] != val_df_err['pred']
val_df_err['error_type'] = ''
val_df_err.loc[(val_df_err['true']==1) & (val_df_err['pred']==0), 'error_type'] = 'FN'
val_df_err.loc[(val_df_err['true']==0) & (val_df_err['pred']==1), 'error_type'] = 'FP'

print('── Errores por tipo ──────────────────────────────────────────')
print(val_df_err[val_df_err['error']]['error_type'].value_counts().to_string())

print('\n── Falsos Negativos (humor no detectado) ─────────────────────')
fn = val_df_err[val_df_err['error_type'] == 'FN'].sort_values('score_si')
for _, row in fn.head(10).iterrows():
    print(f'  score={row["score_si"]:.3f} | {row["tweet"][:80]}')

print('\n── Falsos Positivos (no humor clasificado como humor) ────────')
fp = val_df_err[val_df_err['error_type'] == 'FP'].sort_values('score_si', ascending=False)
for _, row in fp.head(10).iterrows():
    print(f'  score={row["score_si"]:.3f} | {row["tweet"][:80]}')

## ── Sección 10: Evaluación en test ──────────────────────────────────────────

In [ ]:
if 'test' in raw:
    print(f'Evaluando en test ({len(raw["test"])} ejemplos)...')
    test_tweets = list(raw['test'][CFG['text_col']])

    test_preds_raw, test_scores = predict_batch(test_tweets)
    test_scores_np = np.array(test_scores)

    # Aplicar umbral calibrado en validación
    test_preds_final = (test_scores_np >= best_thresh).astype(int)

    print(f'Umbral aplicado: {best_thresh:.2f}')
    unique, counts = np.unique(test_preds_final, return_counts=True)
    print(f'Distribución predicciones: {dict(zip(unique.tolist(), counts.tolist()))}')
else:
    print('ℹ️  No hay split de test')

## ── Sección 11: Guardado del adaptador y resultados ─────────────────────────

In [ ]:
import json

# ── Guardar adaptador LoRA ────────────────────────────────────────────────────
adapter_path = os.path.join(CFG['output_dir'], 'best_adapter')
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ Adaptador guardado: {adapter_path}')

# ── Metadata del experimento ──────────────────────────────────────────────────
meta = {
    'modelo'           : CFG['model_id'],
    'experimento'      : 'Qwen2.5-3B QLoRA con prompt OSTH — Clasificador fine-tuned',
    'lora_r'           : CFG['lora_r'],
    'lora_alpha'       : CFG['lora_alpha'],
    'epochs'           : CFG['epochs'],
    'lr'               : CFG['lr'],
    'max_seq_len'      : CFG['max_seq_len'],
    'threshold'        : float(best_thresh),
    'f1_macro_val'     : float(best_f1),
    'class_weights'    : {str(k): float(v) for k, v in CLASS_WEIGHTS.items()},
    'score_std_val'    : float(val_scores_np.std()),
    'separacion_clases': float(humor_scores.mean() - no_humor_scores.mean()),
    'response_template': RESPONSE_TEMPLATE,
    'siguiente_paso'   : 'Usar este adaptador en el ensamble (reemplaza Qwen base)',
}

meta_path = os.path.join(adapter_path, 'experiment_meta.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print('\n── Metadata del experimento ──────────────────────────────────')
print(json.dumps(meta, indent=2, ensure_ascii=False))

In [ ]:
# ── Guardar predicciones y scores ─────────────────────────────────────────────
if 'test' in raw:
    # CSV de submission (formato concurso)
    submission_df = pd.DataFrame({
        'id'   : range(1, len(test_preds_final) + 1),
        'klass': test_preds_final,
    })
    submission_path = os.path.join(CFG['output_dir'], 'Qwen_OSTH_finetuned_submission.csv')
    submission_df.to_csv(submission_path, index=False)

    # CSV de scores para el ensamble del Experimento 1 actualizado
    scores_df = pd.DataFrame({
        'id'          : range(1, len(test_preds_final) + 1),
        'klass'       : test_preds_final,
        'score_qwen'  : test_scores_np,
    })
    scores_path = os.path.join(CFG['output_dir'], 'Qwen_OSTH_finetuned_scores_test.csv')
    scores_df.to_csv(scores_path, index=False)

# Scores de validación (para re-optimizar pesos del ensamble)
val_scores_df = pd.DataFrame({
    'tweet'      : val_tweets,
    'true'       : val_labels,
    'pred'       : val_preds_calibrated,
    'score_qwen' : val_scores,
})
val_scores_path = os.path.join(CFG['output_dir'], 'Qwen_OSTH_finetuned_scores_val.csv')
val_scores_df.to_csv(val_scores_path, index=False)

print('✅ Archivos guardados:')
if 'test' in raw:
    print(f'   {submission_path}')
    print(f'   {scores_path}')
print(f'   {val_scores_path}')

## ── Sección 12: Cómo integrar este modelo en el ensamble ───────────────────

Una vez entrenado, reemplaza el bloque de carga de Qwen en el notebook `Ensamble_Gemma_RoBERTuito_Qwen_OSTH.ipynb`:

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CÓDIGO PARA COPIAR AL NOTEBOOK DEL ENSAMBLE (Sección 6)               ║
# ╚══════════════════════════════════════════════════════════════════════════╝

ENSAMBLE_INTEGRATION_CODE = '''
# ── Cargar Qwen2.5-3B FINE-TUNED (reemplaza la carga del modelo base) ────────
from peft import PeftModel

QWEN_ADAPTER_PATH = "./qwen3b-osth-humor-qlora/best_adapter"

bnb_config_qwen = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
qwen_base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    quantization_config=bnb_config_qwen, device_map="auto",
)
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_ADAPTER_PATH)
qwen_tokenizer.padding_side = "left"
qwen_model = PeftModel.from_pretrained(qwen_base, QWEN_ADAPTER_PATH)
qwen_model.eval()

# El sistema de inferencia cambia: ahora usamos logits (Sí vs No)
# en lugar de generar JSON con humor_score
QWEN_SI_IDS = qwen_tokenizer.encode("Sí", add_special_tokens=False)
QWEN_NO_IDS = qwen_tokenizer.encode("No", add_special_tokens=False)
QWEN_SI_TOKEN = QWEN_SI_IDS[0]
QWEN_NO_TOKEN = QWEN_NO_IDS[0]

def predict_qwen_finetuned(tweets, batch_size=16):
    """Inferencia Qwen fine-tuned — logits (Sí vs No), sin generación."""
    qwen_model.eval()
    preds, scores = [], []
    for i in range(0, len(tweets), batch_size):
        batch = tweets[i: i + batch_size]
        prompts = [make_prompt(t, include_answer=False) for t in batch]
        inputs = qwen_tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=256,
        ).to(qwen_model.device)
        with torch.no_grad():
            logits = qwen_model(**inputs).logits[:, -1, :]
            si_l = logits[:, QWEN_SI_TOKEN]
            no_l = logits[:, QWEN_NO_TOKEN]
            p_si = torch.softmax(torch.stack([no_l, si_l], dim=1), dim=1)[:, 1]
            preds.extend((si_l > no_l).int().cpu().tolist())
            scores.extend(p_si.cpu().tolist())
    return preds, scores
'''

print('Código de integración para el notebook del ensamble:')
print('─' * 60)
print(ENSAMBLE_INTEGRATION_CODE)

---

## 📋 Checklist post-entrenamiento

Antes de integrar Qwen fine-tuned en el ensamble, verifica:

- [ ] `score_std_val > 0.20` → el modelo discrimina bien (vs ~0.10 del modelo base)
- [ ] `separacion_clases > 0.25` → la distancia entre clases es suficiente
- [ ] `F1 Macro val > 0.78` → el modelo es competitivo individualmente
- [ ] Los FN difíciles (humor implícito/cultural) tienen scores más altos que con el modelo base

## 🔭 Próximo: Experimento 3 — Meta-clasificador con features OSTH

Con los tres clasificadores calibrados, el siguiente paso es un **meta-clasificador** que combina:

```python
# Features del meta-clasificador (LightGBM o LogisticRegression)
X_meta = [
    score_gemma,          # score continuo Gemma fine-tuned
    score_robertuito,     # score continuo RoBERTuito
    score_qwen_finetuned, # score continuo Qwen OSTH fine-tuned
    tipo_oposicion_enc,   # one-hot de normal/anormal|real/irreal|posible/imposible
    oposicion_std_enc,    # one-hot de bueno/malo|vida/muerte|sexo/no-sexo|ninguna
    tiene_punch_line,     # bool
    purview_complejidad,  # longitud del purview como proxy semántico
]
```